# 00 · SHAP — one FTTL version, inside its own environment

In [ ]:
import json
import os
import sys

from collections import OrderedDict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = os.getcwd()
while not os.path.exists(os.path.join(ROOT, "src", "config.py")) and ROOT != os.path.dirname(ROOT):
    ROOT = os.path.dirname(ROOT)
sys.path.insert(0, os.path.join(ROOT, "src"))

import shap_kit_v1 as sk

sk.style()
ENV = sk.env_report(strict=True)

VERSION = "v1"
SPLIT = sk.SPLITS[0]
N_EXPLAIN = 5000        # claims to attribute (TreeSHAP cost is linear in this)
N_BACKGROUND = 500      # interventional reference sample SIZE -- only used when
                        # `shap` is installed here; see sk.has_shap()
SEED = 0
assert SPLIT in sk.SPLITS, (SPLIT, sk.SPLITS)

# real feature name -> anonymised alias ("v1_feat_01", ...). Every figure §5 onward saves is
# drawn TWICE -- real names, then through this mapping -- see the Notes cell at the bottom.
# Plain-json read (sk.load_alias_map): this env cannot import feature_alias.py (3.7+ syntax).
ALIAS_MAP = sk.load_alias_map()

print("\nanalysing {} · split {}".format(VERSION, SPLIT))

## 1 · What this model actually is

In [ ]:
print("--- {} training configuration ---".format(VERSION))
display(pd.Series(sk.TRAINING_CONFIG, name=VERSION).to_frame())

print("\n--- decision rule: {} ---".format(sk.DECISION_RULE["shape"]))
for k in sk.DECISION_RULE:
    print("  {}: {}".format(k, sk.DECISION_RULE[k]))

## 2 · The feature space of *this* repo

In [ ]:
V1_REPO_DIR = ""
MODEL_PATH = ""
assert V1_REPO_DIR or MODEL_PATH, "set V1_REPO_DIR (folder name inside model_repos/real) or MODEL_PATH"
if not MODEL_PATH:
    MODEL_PATH = os.path.join(ROOT, "model_repos", "real", V1_REPO_DIR, "outputs", "fasttracker_xgb.pkl")
print("model: {}".format(MODEL_PATH))
est = sk.load_estimator(MODEL_PATH)
print("  {}".format(type(est).__name__))

FEATURES_PATH = ""
if not FEATURES_PATH:
    FEATURES_PATH = sk.features_csv_path(SPLIT)
print("features: {}".format(FEATURES_PATH))
frame = pd.read_csv(FEATURES_PATH)

ID_COL = "claim_id"
if sk.ID_COLUMN in frame.columns and ID_COL not in frame.columns:
    frame = frame.rename(columns={sk.ID_COLUMN: ID_COL})
if ID_COL in frame.columns:
    ids = frame[ID_COL]
else:
    print("no claim id column found - using positional ids; the saved CSV will NOT join to other artefacts")
    ids = pd.Series(np.arange(len(frame)), name=ID_COL)

X_all = frame.drop([ID_COL], axis=1) if ID_COL in frame.columns else frame
trained = sk.model_feature_names(est)
TRAINED_SOURCE = None          # None -> the pickle named its own columns; a string -> registry
if not trained:
    trained = sk.registry_features()
    TRAINED_SOURCE = sk.registry_features_source()   # goes into the meta with the verdict
    print("  (column list from {})".format(os.path.basename(sk.REGISTRY_PATH)))
missing = [c for c in trained if c not in X_all.columns]
if missing:
    raise RuntimeError("matrix is missing {} model column(s), e.g. {}".format(len(missing), missing[:8]))
extra = [c for c in X_all.columns if c not in trained]
if extra:
    print("set aside {} non-model column(s): {}".format(len(extra), extra[:8]) + (" ..." if len(extra) > 8 else ""))
X_all = X_all[trained]
X_all = sk.align(X_all, est)

for c in X_all.columns:
    if X_all[c].dtype == object:
        X_all[c] = X_all[c].replace({"True": 1, "False": 0, True: 1, False: 0})
non_float = X_all.dtypes[X_all.dtypes != "float64"]
if len(non_float):
    print("casting {} non-float64 column(s) to float64: {}".format(len(non_float), list(non_float.index[:8])))
    X_all = X_all.astype("float64")

print("\n--- {} feature space ---".format(VERSION))
display(sk.feature_summary(X_all).to_frame(VERSION))

In [ ]:
profile = sk.describe_features(X_all)
print("per-feature profile — sorted by cardinality (full table in `profile`)")
display(profile.sort_values("n_unique", ascending=False).head(25))

fams = profile[profile["kind"] == "binary"]["family"].value_counts()
if (fams > 1).any():
    print("\none-hot families (raw columns that share a prefix) — NOT collapsed, only counted:")
    display(fams[fams > 1].head(15).to_frame("n_columns"))

## 3 · Scores and the fast-track region

v1's rule is **segmented on vehicle mobility**: `score > 0.75` fast-tracks an *immobile* car,
`score > 0.85` a *mobile* one, both STRICT `>`. **The segmenting column is unrecoverable** —
mobility is in neither the v1 training matrix nor the raw dataset (confirmed 2026-08-27), so the
segmented rule can never be applied per row here and there is nothing left to join.

Everything below therefore uses the conservative `s > 0.85`, and every claim in `(0.75, 0.85]` is
counted as garage even though an immobile car there *was* fast-tracked in production. That band is
a **limitation to state in the thesis**, not a gap to fill later: inside it a model-scrapped claim
and a garage-verified-then-scrapped one have different label provenance and cannot be told apart.
Its size is recorded with the attributions as `n_in_band_unresolved` (§11).

In [ ]:
# v1's rule is SEGMENTED on vehicle mobility: `score > 0.75` fast-tracks an IMMOBILE car,
# `score > 0.85` a MOBILE one, both comparisons STRICT `>` (sk.DECISION_RULE, read off the
# serving code). The segmenting column is UNRECOVERABLE -- mobility is in neither the v1
# training matrix nor the raw dataset (confirmed 2026-08-27) -- so there is nothing left to join
# and the rule can NEVER be applied per row here. This notebook uses the CONSERVATIVE `s > 0.85`:
# certain fast-tracks only, with every claim in (0.75, 0.85] counted as garage even though an
# immobile car there WAS fast-tracked in production.
#
# That band is a thesis LIMITATION to state, not a gap to fill: inside it a MODEL-scrapped claim
# (immobile, fast-tracked at 0.75) and a GARAGE-VERIFIED-then-scrapped one (mobile, sent to the
# garage and written off there) have different label provenance and are indistinguishable. Its
# size travels with the attributions as `n_in_band_unresolved` (§11), not only in a print here.
scores = est.predict_proba(X_all)[:, 1]

TAUS = sk.DECISION_RULE["thresholds"]                       # immobile 0.75 / mobile 0.85
TAU_IMMOBILE, TAU_MOBILE = float(TAUS["immobile"]), float(TAUS["mobile"])
BAND_TXT = "({}, {}]".format(TAU_IMMOBILE, TAU_MOBILE)      # open left, CLOSED right: at exactly
TAU = TAU_MOBILE                                            #   0.85 the immobile car is fast-tracked, the mobile one is not

above = scores > TAU_MOBILE                                 # STRICT >, and the HIGHER cutoff only
unresolved = (scores > TAU_IMMOBILE) & (scores <= TAU_MOBILE)
tau_note = ("segmented on mobility {} (STRICT >); the mobility column is unrecoverable, so "
            "`above` is the conservative s > {} and the {} claims in {} are PERMANENTLY "
            "undecided -- an immobile car in that band IS fast-tracked in production and is "
            "counted as garage here".format(
                dict(TAUS), TAU_MOBILE, int(unresolved.sum()), BAND_TXT))

print("τ = {} immobile / {} mobile   ({})".format(TAU_IMMOBILE, TAU_MOBILE, tau_note))
print("scored {} claims · above τ: {} ({:.2%})".format(
    len(scores), int(above.sum()), above.mean()))
print("  s <= {}: {} claims (garage in BOTH segments)".format(
    TAU_IMMOBILE, int((scores <= TAU_IMMOBILE).sum())))
print("  {} < s <= {}: {} claims (mobility decides -- UNDECIDED, and unrecoverably so)".format(
    TAU_IMMOBILE, TAU_MOBILE, int(unresolved.sum())))
print("  s >  {}: {} claims (fast-track in BOTH segments)".format(TAU_MOBILE, int(above.sum())))
print("score range {:.4f} – {:.4f}, median {:.4f}".format(
    scores.min(), scores.max(), np.median(scores)))


def tau_label(ax, t_, text, y_frac=0.9):
    """Annotate a cutoff line INSIDE the axes -- flipped to the left when t sits near the
    right edge, which is what pushed the mobile label off the canvas."""
    xlo, xhi = ax.get_xlim()
    ha = "left" if (xhi - t_) / float(xhi - xlo) > 0.22 else "right"
    ax.text(t_, ax.get_ylim()[1] * y_frac, (" " + text) if ha == "left" else (text + " "),
            color=sk.RED, fontsize=8, va="top", ha=ha)


fig, ax = plt.subplots(figsize=(sk.FIG_W, 3.2))
ax.hist(scores, bins=80, color=sk.BLUE)
ax.set_yscale("log")
ax.axvspan(TAU_IMMOBILE, TAU_MOBILE, color=sk.GREY, alpha=0.18, lw=0)
for t_, lab, y_frac in ((TAU_IMMOBILE, "immobile", 0.9), (TAU_MOBILE, "mobile", 0.06)):
    ax.axvline(t_, color=sk.RED, lw=1.4, ls="--")
    tau_label(ax, t_, "τ {} = {}".format(lab, t_), y_frac)
ax.set_xlabel("model_{}_score".format(VERSION))
ax.set_ylabel("claims (log)")
ax.set_title("{} — score distribution and the TWO fast-track cutoffs "
             "(shaded: {} claims where mobility decides and cannot be recovered)".format(
                 VERSION, int(unresolved.sum())), fontsize=10)
fig.tight_layout()

sk.save(fig, "v1_03a_score_distribution")
plt.show()

## 4 · SHAP values

In [ ]:
rng = np.random.RandomState(SEED)
n = len(X_all)
exp_idx = rng.choice(n, size=min(N_EXPLAIN, n), replace=False)
X = X_all.iloc[exp_idx]
s = scores[exp_idx]
row_ids = ids.iloc[exp_idx].values
above_x = above[exp_idx]        # per-row decision from §3 (segmented τ), on the explained rows

# BOTH backends in one pass, exactly as 00_SHAP.ipynb does for v2/v3: interventional needs a
# background sample, tree_path_dependent measures against each tree's own cover and takes
# None. Without `shap` this env can only do the latter -- the notebook still runs, and 11
# writes that result under "_native" so it never occupies the interventional slot the
# cross-version notebook reads.
SHAP_VERSION = sk.has_shap()
print("shap in this env: {}".format(
    SHAP_VERSION or "NOT INSTALLED -- tree_path_dependent only (see src/envs/v1/check_shap.py)"))

# Background drawn from rows NOT being explained where the split is large enough for that.
bg_pool = np.setdiff1d(np.arange(n), exp_idx)
if len(bg_pool) < N_BACKGROUND:
    bg_pool = np.arange(n)
bg_idx = rng.choice(bg_pool, size=min(N_BACKGROUND, len(bg_pool)), replace=False)
background = X_all.iloc[bg_idx]

wanted = [("interventional", background), ("tree_path_dependent", None)] if SHAP_VERSION \
         else [("tree_path_dependent", None)]

att_by_backend = OrderedDict()
for _name, _bg in wanted:
    _a = sk.compute(est, X, background=_bg)
    # The label decides the FILENAME in 11. A mislabelled file is the one error nothing
    # downstream can catch, so refuse rather than trust the request.
    assert _a.perturbation == _name, (_name, _a.perturbation)
    print("{}".format(_a))
    sk.check_additivity(_a, est)
    att_by_backend[_name] = _a

ACTIVE_BACKEND = "interventional" if SHAP_VERSION else "tree_path_dependent"
att = att_by_backend[ACTIVE_BACKEND]

# Fail fast, once, HERE -- not mid-figure several cells down -- if ALIAS_MAP (§0) does not cover
# every column this att actually carries (a stale map: registry rebuilt, alias map not).
# att_alias is reused by every figure cell from §5 on.
att_alias = att.relabel(ALIAS_MAP)

print("\nfigures below use: {} (background {})".format(
    ACTIVE_BACKEND, len(background) if ACTIVE_BACKEND == "interventional" else 0))

print("\ntop features by mean|SHAP| ({} - {} - {}):".format(VERSION, SPLIT, ACTIVE_BACKEND))
display(att.mean_abs.head(15).round(5).to_frame("mean_abs"))

## 5 · Input distributions, below vs above the cutoff

In [ ]:
DIST_FEATURES = att.top(9)
fig = sk.plot_distributions(X, DIST_FEATURES, scores=s, above=above_x, ncols=3,
                            title="{} — input distributions of the top SHAP drivers".format(VERSION))
sk.save(fig, "v1_05a_input_distributions")
plt.show()

# SAME distributions, alias labels -- an independent figure/save, not this one relabelled (see
# Notes at the bottom). Real-name PNG is company-laptop/internal only; alias_ is the only
# version safe to leave the machine.
X_alias = X.rename(columns=ALIAS_MAP)
DIST_FEATURES_ALIAS = [ALIAS_MAP[f] for f in DIST_FEATURES]
fig_alias = sk.plot_distributions(
    X_alias, DIST_FEATURES_ALIAS, scores=s, above=above_x, ncols=3,
    title="{} — input distributions of the top SHAP drivers (aliased)".format(VERSION))
sk.save(fig_alias, "alias_v1_05a_input_distributions")
plt.show()

## 6 · Global interpretability

In [ ]:
fig = sk.plot_bar(att, top_n=20, title="{} — global importance (mean |SHAP|, {})".format(VERSION, ACTIVE_BACKEND))
sk.save(fig, "v1_06a_shap_bar")
plt.show()

fig_alias = sk.plot_bar(
    att_alias, top_n=20,
    title="{} — global importance (mean |SHAP|, {}) (aliased)".format(VERSION, ACTIVE_BACKEND))
sk.save(fig_alias, "alias_v1_06a_shap_bar")
plt.show()

In [ ]:
fig = sk.plot_beeswarm(att, top_n=20, title="{} — SHAP beeswarm (signed, {})".format(VERSION, ACTIVE_BACKEND))
sk.save(fig, "v1_06b_shap_beeswarm")
plt.show()

fig_alias = sk.plot_beeswarm(
    att_alias, top_n=20,
    title="{} — SHAP beeswarm (signed, {}) (aliased)".format(VERSION, ACTIVE_BACKEND))
sk.save(fig_alias, "alias_v1_06b_shap_beeswarm")
plt.show()

In [ ]:
fig = sk.plot_beeswarm_abs(att, top_n=20,
                           title="{} — |SHAP| spread, coloured by mean |SHAP|".format(VERSION))
sk.save(fig, "v1_06c_shap_beeswarm_abs")
plt.show()

fig_alias = sk.plot_beeswarm_abs(
    att_alias, top_n=20,
    title="{} — |SHAP| spread, coloured by mean |SHAP| (aliased)".format(VERSION))
sk.save(fig_alias, "alias_v1_06c_shap_beeswarm_abs")
plt.show()

## 7 · Interactions — what the vertical dispersion is

In [ ]:
strength = None
assoc = sk.feature_association(X, method="spearman", features=att.top(30))
print("strongest column associations in X (Spearman) — a property of the data, NOT of the model:")
display(sk.top_associated_pairs(assoc, k=10).round(3))

## 8 · Dependence — the value → contribution shape

In [ ]:
DEP_FEATURES = [f for f in att.top(12)
                if pd.api.types.is_numeric_dtype(X[f]) and X[f].nunique() > 4][:4] or att.top(3)
for feat in DEP_FEATURES:
    fig = sk.plot_dependence(att, feat, interaction="auto",
                             title="{} — dependence: {}".format(VERSION, feat))
    sk.save(fig, "v1_08a_dependence_{}".format(feat))
    plt.show()

    # SAME dependence relation, alias labels AND alias filename -- the real feature name must
    # not leak through the filename either (unlike the section-level figures above, this one
    # embeds it). interaction="auto" on att_alias independently re-picks the partner column --
    # the correlation _pick_interaction uses is over VALUES, unchanged by relabel, so it always
    # lands on the same underlying column and just reports it under its alias name; nothing
    # needs to be translated by hand. See Notes at the bottom.
    alias_feat = ALIAS_MAP[feat]
    fig_alias = sk.plot_dependence(
        att_alias, alias_feat, interaction="auto",
        title="{} — dependence: {} (aliased)".format(VERSION, alias_feat))
    sk.save(fig_alias, "alias_v1_08a_dependence_{}".format(alias_feat))
    plt.show()

## 9 · Local interpretability — individual claims

In [ ]:
# τ here is the single conservative cutoff 0.85 (§3): the segmented rule needs a mobility
# column that no longer exists, so "just above/below τ" is nearest 0.85 and nothing crosses.
# Under the rule as SERVED they could: the lowest fast-tracked score (immobile, just over 0.75)
# sits BELOW the highest garaged score (mobile, up to 0.85). That ordering is unobservable here.
picks = []
if above_x.any():
    idx_above = np.where(above_x)[0]
    picks.append(("strongest fast-track", int(idx_above[np.argmax(s[idx_above])])))
    picks.append(("just above τ", int(idx_above[np.argmin(s[idx_above])])))
idx_below = np.where(~above_x)[0]
if len(idx_below):
    picks.append(("just below τ", int(idx_below[np.argmax(s[idx_below])])))

for label, row in picks:
    print("{}: row {}  ·  claim {}  ·  score {:.4f}".format(label, row, row_ids[row], s[row]))
    fig = sk.plot_waterfall(att, row=row, top_n=12,
                            label="{} — {}  (score {:.4f})".format(VERSION, label, s[row]))
    sk.save(fig, "v1_09a_waterfall_{}".format(label.replace(" ", "_")))
    plt.show()

    # SAME claim, alias feature labels (att_alias from §4). `label` is a claim-selection tag
    # ("just above τ"), never a feature name, so the filename only needs the alias_ prefix -- the
    # content (each bar's feature name) is what needed relabelling. See Notes at the bottom.
    fig_alias = sk.plot_waterfall(
        att_alias, row=row, top_n=12,
        label="{} — {}  (score {:.4f}) (aliased)".format(VERSION, label, s[row]))
    sk.save(fig_alias, "alias_v1_09a_waterfall_{}".format(label.replace(" ", "_")))
    plt.show()

In [ ]:
for label, row in picks:
    fig = sk.plot_force(att, row=row, top_n=10,
                        label="{} — force: {}  (score {:.4f})".format(VERSION, label, s[row]))
    sk.save(fig, "v1_09b_force_{}".format(label.replace(" ", "_")))
    plt.show()

    fig_alias = sk.plot_force(
        att_alias, row=row, top_n=10,
        label="{} — force: {}  (score {:.4f}) (aliased)".format(VERSION, label, s[row]))
    sk.save(fig_alias, "alias_v1_09b_force_{}".format(label.replace(" ", "_")))
    plt.show()

### 9b · Averaged over many claims — the band immediately around τ

In [ ]:
N_NEAR_ABOVE = 50
N_NEAR_BELOW = 50

above_mask = above_x
below_mask = ~above_mask

near_above_idx = np.where(above_mask)[0][np.argsort(s[above_mask])[:N_NEAR_ABOVE]]
near_below_idx = np.where(below_mask)[0][np.argsort(-s[below_mask])[:N_NEAR_BELOW]]

near_labels = np.full(len(s), np.nan, dtype=object)
near_labels[near_below_idx] = "below τ"
near_labels[near_above_idx] = "above τ"
near_bands = pd.Series(pd.Categorical(near_labels, categories=["below τ", "above τ"], ordered=True))

print("near-cutoff band: {} claims below τ (closest) vs {} claims above τ (closest) — "
      "requested {}/{}".format(len(near_below_idx), len(near_above_idx), N_NEAR_BELOW, N_NEAR_ABOVE))

fig, near_table = sk.plot_band_bars(
    att, near_bands, top_n=12,
    title="{} — mean |SHAP|, {} closest-below vs {} closest-above τ".format(
        VERSION, len(near_below_idx), len(near_above_idx)))
sk.save(fig, "v1_09ba_near_tau_band_comparison")
plt.show()
display(near_table.round(4))

# SAME bands, alias feature labels (att_alias from §4). near_table's printed values are
# unaffected by aliasing (same convention as every other notebook here: only the SAVED figure
# needs the alias pass, not a displayed cell-output table). See Notes at the bottom.
fig_alias, _ = sk.plot_band_bars(
    att_alias, near_bands, top_n=12,
    title="{} — mean |SHAP|, {} closest-below vs {} closest-above τ (aliased)".format(
        VERSION, len(near_below_idx), len(near_above_idx)))
sk.save(fig_alias, "alias_v1_09ba_near_tau_band_comparison")
plt.show()

### 9c · The same claims, kept individual — not averaged

In [ ]:
TOP_N_FEATURES_9C = 12

feats_9c = att.top(TOP_N_FEATURES_9C)
feat_j = [att.features.index(f) for f in feats_9c]

above_order = near_above_idx[np.argsort(-s[near_above_idx])]
below_order = near_below_idx[np.argsort(-s[near_below_idx])]
above_scores = s[above_order]
below_scores = s[below_order]

if len(above_order) == 0 or len(below_order) == 0:
    print("one side is empty (above={}, below={}) — nothing to plot side by side.".format(
        len(above_order), len(below_order)))
else:
    above_phi = att.phi[np.ix_(above_order, feat_j)]
    below_phi = att.phi[np.ix_(below_order, feat_j)]
    vmax = float(np.abs(np.concatenate([above_phi, below_phi])).max())

    def _near_tau_heatmap_fig(aliased) -> None:
        """Two independent figures, real name then alias -- see Notes at the bottom. The phi
        values are identical either way; only the x-tick labels (feature names) change."""
        labels = [ALIAS_MAP[f] for f in feats_9c] if aliased else list(feats_9c)
        fig, axes = plt.subplots(
            1, 2, figsize=(sk.FIG_W + 2.6, 0.16 * max(len(above_order), len(below_order)) + 2.4))
        for ax, phi_mat, scores_, label in (
            (axes[0], above_phi, above_scores, "above τ  (n={})".format(len(above_order))),
            (axes[1], below_phi, below_scores, "below τ  (n={})".format(len(below_order))),
        ):
            im = ax.imshow(phi_mat, aspect="auto", cmap=sk.CMAP, vmin=-vmax, vmax=vmax)
            ax.set_xticks(np.arange(len(labels)))
            ax.set_xticklabels(labels, rotation=90, fontsize=7)
            ax.set_yticks(np.arange(len(scores_)))
            ax.set_yticklabels(["{:.4f}".format(v) for v in scores_], fontsize=5.5)
            ax.set_title(label, fontsize=9)
        axes[0].set_ylabel("claim score (descending)")

        fig.colorbar(im, ax=list(axes), fraction=0.025, pad=0.02, label="SHAP value  (log-odds)")
        fig.suptitle("{} — individual near-τ claims, φ by feature ({} features){}".format(
            VERSION, len(feats_9c), " (aliased)" if aliased else ""))
        prefix = "alias_" if aliased else ""
        sk.save(fig, "{}v1_09ca_near_tau_individual_heatmap".format(prefix))
        plt.show()

    _near_tau_heatmap_fig(False)
    _near_tau_heatmap_fig(True)

## 10 · Inside the fast-track region — mean |SHAP| by score band

In [ ]:
bands = sk.score_bands(s, TAU, quantiles=(0.5, 0.9, 0.99), above=above_x)
print(bands.value_counts().to_frame("claims"))

fig, band_table = sk.plot_band_bars(att, bands, top_n=12,
                                    title="{} — mean |SHAP| by score band".format(VERSION))
sk.save(fig, "v1_10a_band_bars")
plt.show()
display(band_table.round(4))

# SAME bands, alias feature labels (att_alias from §4). See Notes at the bottom.
fig_alias, _ = sk.plot_band_bars(
    att_alias, bands, top_n=12,
    title="{} — mean |SHAP| by score band (aliased)".format(VERSION))
sk.save(fig_alias, "alias_v1_10a_band_bars")
plt.show()

## 11 · Save the attributions

In [ ]:
def save_attribution(a):
    """Write one backend's phi + sidecar meta. The backend decides the filename: the
    canonical no-suffix name is the interventional slot 00_shap_attribution.ipynb's
    interventional runs read, "_native" is what its path_dependent run reads."""
    suffix = sk.out_suffix(a.perturbation)
    out_path = sk.attributions_csv_path(SPLIT, suffix)
    out_dir = os.path.dirname(out_path)
    if not os.path.isdir(out_dir):
        os.makedirs(out_dir)

    frame_out = a.frame(id_values=row_ids, id_col=ID_COL)
    frame_out.to_csv(out_path, index=False)

    meta = OrderedDict([
        ("version", VERSION),
        ("split", SPLIT),
        ("model_path", str(MODEL_PATH)),
        ("features_path", str(FEATURES_PATH)),
        ("estimator", type(est).__name__),
        ("estimator_params", OrderedDict(
            (k, v if isinstance(v, (int, float, bool, str)) or v is None else str(v))
            for k, v in sorted(est.get_params().items()))),
        # the shared status word, same vocabulary as shap_kit.feature_order — X went
        # through sk.align() above, so this records that the check RAN. v1 can
        # legitimately say "unverified": xgboost 0.72 may only expose f0/f1/... names.
        ("feature_order", sk.feature_order(est, a.features,
                                           trained=(trained if TRAINED_SOURCE else None),
                                           trained_source=TRAINED_SOURCE)),
        ("backend", a.backend),
        ("perturbation", a.perturbation),
        ("model_output", "raw"),
        ("note", a.note),
        ("n_rows", int(a.phi.shape[0])),
        ("n_features", int(a.phi.shape[1])),
        ("feature_names", a.features),
        ("background_n", int(len(background)) if a.perturbation == "interventional" else 0),
        ("seed", SEED),
        ("decision_rule", OrderedDict([("shape", sk.DECISION_RULE["shape"]),
                                      ("segment_by", sk.DECISION_RULE["segment_by"])])),
        ("tau_used", OrderedDict([("immobile", TAU_IMMOBILE), ("mobile", TAU_MOBILE)])),
        # No segmenting column and no source to name it from: mobility is unrecoverable (§3).
        # Kept in the meta so a reader of the file sees which path produced it without opening
        # the notebook -- and so a future file carrying a real column is distinguishable.
        ("tau_segment_column", None),
        ("tau_segment_source", None),
        ("tau_applied_per_row", False),
        # WHICH cutoff produced `above`, spelled out: with two taus in `tau_used` a bare count
        # is ambiguous, and this one is the MOBILE (higher) cutoff on every row.
        ("tau_rule_applied", "s > {} (mobile cutoff only; the immobile 0.75 needs a column "
                             "that no longer exists)".format(TAU_MOBILE)),
        # ... and over WHICH rows. §4 attributes a SEED-sampled N_EXPLAIN subset, so the split
        # count and the count for the rows actually in this CSV are different numbers. Both,
        # or a reader silently compares a rate on 5000 rows against a numerator on the split.
        # The score axis in THREE buckets, on both populations, so the counts are checkable:
        #   n_scored = n_below_band + n_in_band_unresolved + n_above_tau
        #   n_rows   = the same three "_explained" counts
        # and the fast-track count production actually made is bracketed, never asserted:
        #   n_above_tau  <=  true fast-tracks  <=  n_above_tau + n_in_band_unresolved
        ("n_scored", int(len(scores))),                                 # the whole split
        ("n_below_band", int((scores <= TAU_IMMOBILE).sum())),          # garage in BOTH segments
        ("n_below_band_explained", int((s <= TAU_IMMOBILE).sum())),
        ("n_above_tau", int(above.sum())),                              # fast-track in BOTH
        ("n_above_tau_explained", int(above_x.sum())),                  # the rows IN this file
        # The SIZE of the thesis limitation, recorded with the file rather than left in a
        # print: the claims the segmented rule WOULD decide on mobility and this run cannot.
        # An immobile car here was fast-tracked in production and is counted as garage above.
        # This is never zero for v1 -- the column it would need does not exist any more.
        ("n_in_band_unresolved", int(unresolved.sum())),                # mobility would decide
        ("n_in_band_unresolved_explained", int(unresolved[exp_idx].sum())),
        ("band_unresolved", BAND_TXT),
        ("tau_note", tau_note),
        ("base_value", float(np.mean(a.base))),
        ("env", ENV),
    ])
    meta_path = os.path.splitext(out_path)[0] + "_meta.json"
    with open(meta_path, "w", encoding="utf-8") as fh:
        fh.write(json.dumps(meta, indent=2))

    run = "interventional_*" if suffix == "" else "path_dependent"
    print("[{}/{}] wrote {}".format(a.backend, a.perturbation, out_path))
    print("{:>12}{}".format("", meta_path))
    print("{:>12}-> the {} run in 00_shap_attribution.ipynb".format("", run))
    return out_path


written = [save_attribution(a) for a in att_by_backend.values()]

print("\nconvert each to parquet in the analysis .venv before 00_shap_attribution.ipynb reads\n"
      "them -- v1_csv_to_parquet.py does NOT cover detection/shap/:")
for _p in written:
    print("  pd.read_csv(r\"{}\").to_parquet(r\"{}\", index=False)".format(
        _p, os.path.splitext(_p)[0] + ".parquet"))

## Notes — what would make the figures above wrong

- **Every feature-labelled figure from §5 on is saved TWICE, real name then alias.** §0 builds
  `ALIAS_MAP` via `sk.load_alias_map()` — the plain-json equivalent of
  `feature_alias.load_version("v1")["real_to_alias"]` (`src/feature_alias.py`), read directly with
  stdlib `json` because this env cannot import that module (3.7+ syntax: `from __future__ import
  annotations`, f-strings). §4 fails fast (`att_alias = att.relabel(ALIAS_MAP)`, which raises if
  the map does not cover every column this run's attribution carries) before any figure, not
  mid-way through one, and `att_alias` is reused by every figure cell from §5 on.
    - Where the plot takes an `Attribution`, `att.relabel(mapping)` (`shap_kit_v1.py`, same method
      name and contract as the v2/v3 twin in `shap_kit.py`) is the one seam — it returns a copy
      with `.X`'s columns renamed (`phi` is positional, untouched), so `plot_bar` / `plot_beeswarm`
      / `plot_beeswarm_abs` / `plot_dependence` / `plot_waterfall` / `plot_force` /
      `plot_band_bars` all inherit the alias for free.
    - Where the plot takes a standalone `X` (§5's `plot_distributions`), the same rename is done
      inline (`X.rename(columns=...)`) since there is no shared object to carry it.
    - Each pair is TWO INDEPENDENT figures (two separate `plot_*`/`plt.subplots` calls, two
      separate `sk.save`s) — never one figure relabelled and re-saved — so neither figure's layout
      is sized around the other's label widths.
    - §8's dependence plots are the one case where the real feature name sits IN THE FILENAME
      (`08a_dependence_<feat>`), so the alias save also swaps the filename's feature name for its
      alias, not just an `alias_` prefix on the real one — a plain prefix there would still leak
      the real name through the filename. `interaction="auto"` on `att_alias` independently
      re-picks the partner column (the correlation `_pick_interaction` uses is over VALUES,
      unchanged by relabel), so nothing needs to be translated by hand there.
    - §3's score-distribution (`v1_03a`) and §11's saved attribution CSVs carry no per-feature
      label at all (the former plots the model's output, not an input; the latter is pipeline
      data the next real-data step reads with real names on purpose) — neither gets an alias twin.
      §7 here is data-association tables only (`display()`, no saved figure), same convention as
      every other notebook: only a SAVED figure needs the alias pass, not a printed cell output.
  Real-name PNGs are company-laptop / internal use only; the `alias_`-prefixed ones are the only
  version safe to leave the machine (thesis, review, anywhere the real Allianz column names must
  not go). `figures/` is NOT gitignored, so the real-name PNGs are stageable — do not `git add`
  them. The mapping itself (`features/registry/feature_alias_map.json`) is built once on the
  company laptop by `features/build_feature_alias.py` and never reaches git.
- **This is the SUBSET twin of `00_SHAP.ipynb`.** No §6b (shap's native plots need `Explanation`,
  shap 0.36+) and no §7 interaction-strength route (`interaction_values`/TreeSHAP interaction
  matrix) — `shap_kit_v1.py` only carries `_pick_interaction`, a correlation proxy, which is what
  §8's `interaction="auto"` uses. `has_shap()` still lets `compute()` use real shap (0.35.0) when
  installed, ahead of the booster's own `pred_contribs` fallback.
- **τ is the conservative `s > 0.85` only** (§3) — the segmented rule needs a `mobility` column
  that exists in neither the training matrix nor the raw dataset, so it can never be applied per
  row. The `(0.75, 0.85]` band is a permanent thesis limitation, not a to-do — see §3's markdown
  and `n_in_band_unresolved` in the §11 meta.
- **Never collapse raw column names to compare with another version.** Cross-version
  correspondence comes only from the hand-confirmed mapping (`features/check_overlap.py` →
  `features/feature_overlap.json`) — never from name equality.